# SWE409 – Introduction to AI and Data Science
## Pandas Practical Exercises – Part 2 of 3: Indexes, Combining Tables & Grouping
**Lecturer:** Engr. Kang Modest Ekome  |  **Reference notes:** *Lecture 2 – Pandas for Data Analytics (PDF)*

| | |
|---|---|
| **Student Name:** | |
| **Speciality:** | |
| **Matricule / Date:** | |

> Fill in your details in the cell below **before** you start. Your name must appear when you submit.

In [1]:
# ---- STUDENT INFORMATION (fill in) ----
student_name = "Takang Tracy"   # Student Name: e.g. "Ngwa Divine"
speciality   = "Software Engineering"   # Speciality:   e.g. "Software Engineering"
print("Student Name:", student_name)
print("Speciality  :", speciality)

Student Name: Takang Tracy
Speciality  : Software Engineering


### How to work through this notebook
1. Read the **scenario**, then answer the questions **in order** – they get progressively harder.
2. Every question has a **TASK**, a **HINT** (with the lecture slide to revise) and a code cell with comments telling you what to do.
3. Replace each `None  # <-- ...` with your own pandas code. **Do not change the variable names** – they are checked by the lecturer.
4. Run each cell (`Shift + Enter`) and check the output makes sense before moving on.
5. Cells that are already filled in (data set-up) must be run but **not edited**.
6. Slide numbers refer to the PDF page numbers of *Lecture 2 – Pandas for Data Analytics (PDF)*.

**Quick revision map (slide numbers in the lecture PDF)**

| Topic | Slides |
|---|---|
| Index as row labels, `loc` / `iloc` | 28, 30, 35 |
| Hierarchical (Multi) index & selecting levels | 222 – 225 |
| Concatenating tables – `pd.concat()` | 217 |
| Merging on keys – `pd.merge()` | 218 |
| Joining on the index – `.join()` | 219 |
| Describing a table: `info`, `describe` | 137 – 145 |
| `value_counts`, `unique` | 149 – 154 |
| `groupby` + aggregation | 127 – 131 |

### 🏢 The big scenario
You now work for a **digital-services start-up with offices in Douala and Buea**. Management keeps its data in many small tables (staff, salaries, managers, branch sales, mobile-money agents, cocoa deliveries). Your job is to **organise indexes, combine the tables, and summarise** them with grouping.

In [2]:
# (provided) run first
import pandas as pd
import numpy as np

---
## Part A – Choosing the right index
HR gives you the class list of the company's **intern trainees**. By default pandas numbers the rows 0, 1, 2… but the *matricule* is a much better row label.

In [3]:
interns = pd.DataFrame({
    "matricule": ["INT01", "INT02", "INT03", "INT04", "INT05", "INT06"],
    "name":      ["Ngwa", "Aissatou", "Tchoua", "Mbarga", "Fotso", "Ekema"],
    "office":    ["Douala", "Buea", "Douala", "Buea", "Douala", "Buea"],
    "score":     [14.5, 16.0, 12.5, 17.5, 11.0, 15.0]})
interns

,matricule,name,office,score
0,INT01,Ngwa,Douala,14.5
1,INT02,Aissatou,Buea,16.0
2,INT03,Tchoua,Douala,12.5
3,INT04,Mbarga,Buea,17.5
4,INT05,Fotso,Douala,11.0
5,INT06,Ekema,Buea,15.0


### Question 1  *(2 marks)*
**TASK:** Make `matricule` the index of `interns` and store the result in `by_mat`. Then show the intern **INT04** with `.loc` (`int04`).

> 💡 **HINT:** `df.set_index('column')` moves a column into the index (it returns a NEW table). Then `.loc['INT04']` finds the row by label (slide 30).

In [4]:
# ===== Q1 =====
# TASK: Make `matricule` the index of `interns` and store the result in `by_mat`. Then show the intern **INT04** with 
# by_mat -> set_index on the matricule column
# int04 -> .loc with the label 'INT04'
by_mat = interns.set_index('matricule') 
int04 = int04 = by_mat.loc['INT04']

# show your answers
print('by_mat'); display(by_mat)
print('int04'); display(int04)

by_mat


,name,office,score
matricule,,,
INT01,Ngwa,Douala,14.5
INT02,Aissatou,Buea,16.0
INT03,Tchoua,Douala,12.5
INT04,Mbarga,Buea,17.5
INT05,Fotso,Douala,11.0
INT06,Ekema,Buea,15.0


int04


name      Mbarga
office      Buea
score       17.5
Name: INT04, dtype: object

### Question 2  *(2 marks)*
**TASK:** Undo it: (a) move the index of `by_mat` back to an ordinary column (`back_to_col`); (b) throw the index away and get a clean 0,1,2… index instead (`fresh_index`).

> 💡 **HINT:** `reset_index()` brings the index back as a column; `reset_index(drop=True)` discards it.

In [ ]:
# (provided) so you do not depend on your previous answer
by_mat = interns.set_index("matricule")

In [5]:
# ===== Q2 =====
# TASK: Undo it: (a) move the index of `by_mat` back to an ordinary column (`back_to_col`); (b) throw the index away a
# back_to_col -> reset_index()
# fresh_index -> reset_index with drop=True
back_to_col =  by_mat.reset_index()  
fresh_index = by_mat.reset_index(drop=True)

# show your answers
print('back_to_col'); display(back_to_col)
print('fresh_index'); display(fresh_index)

back_to_col


,matricule,name,office,score
0,INT01,Ngwa,Douala,14.5
1,INT02,Aissatou,Buea,16.0
2,INT03,Tchoua,Douala,12.5
3,INT04,Mbarga,Buea,17.5
4,INT05,Fotso,Douala,11.0
5,INT06,Ekema,Buea,15.0


fresh_index


,name,office,score
0,Ngwa,Douala,14.5
1,Aissatou,Buea,16.0
2,Tchoua,Douala,12.5
3,Mbarga,Buea,17.5
4,Fotso,Douala,11.0
5,Ekema,Buea,15.0


---
## Part B – Hierarchical (Multi) Index: mobile-money agents
Two cities × three services = six rows of **weekly commission (thousand FCFA)**. A *MultiIndex* lets you keep both labels in the index. Run the cell to get the raw lists.

In [6]:
cities   = ["Douala", "Douala", "Douala", "Yaounde", "Yaounde", "Yaounde"]
services = ["MoMo", "Airtime", "Transfers", "MoMo", "Airtime", "Transfers"]
commission = [[82, 90], [40, 44], [65, 70], [75, 78], [38, 36], [60, 66]]   # [week 1, week 2]

### Question 3  *(2 marks)*
**TASK:** Zip `cities` and `services` into a list of tuples (`pairs`) and build a MultiIndex `idx` from it.

> 💡 **HINT:** Slides 222–223: `pd.MultiIndex.from_tuples(list_of_tuples)`. `list(zip(a, b))` pairs two lists element by element.

In [7]:
# ===== Q3 =====
# TASK: Zip `cities` and `services` into a list of tuples (`pairs`) and build a MultiIndex `idx` from it.
# pairs -> list of (city, service) tuples
# idx   -> MultiIndex built from the tuples
pairs = list(zip(cities, services))
 #idx = None  # <-- replace None with your cod

# show your answers
print('pairs'); display(pairs)
 #print('idx'); display(idx)

pairs


[('Douala', 'MoMo'),
 ('Douala', 'Airtime'),
 ('Douala', 'Transfers'),
 ('Yaounde', 'MoMo'),
 ('Yaounde', 'Airtime'),
 ('Yaounde', 'Transfers')]

In [15]:
# spairs = list(zip(cities, services))
idx = pd.MultiIndex.from_tuples(pairs)

# show your answers
print('pairs'); display(pairs)
print('idx'); display(idx)

pairs


[('Douala', 'MoMo'),
 ('Douala', 'Airtime'),
 ('Douala', 'Transfers'),
 ('Yaounde', 'MoMo'),
 ('Yaounde', 'Airtime'),
 ('Yaounde', 'Transfers')]

idx


MultiIndex([( 'Douala',      'MoMo'),
            ( 'Douala',   'Airtime'),
            ( 'Douala', 'Transfers'),
            ('Yaounde',      'MoMo'),
            ('Yaounde',   'Airtime'),
            ('Yaounde', 'Transfers')],
           )

### Question 4  *(2 marks)*
**TASK:** Create the DataFrame `agents` from `commission`, with `idx` as the index and the columns `"week1"` and `"week2"`. Then name the two index levels `"city"` and `"service"`.

> 💡 **HINT:** `pd.DataFrame(data, index=..., columns=[...])`. Level names are set with `df.index.names = [...]` (slide 222).

In [16]:
# (provided) in case your previous answers are not finished
pairs = list(zip(cities, services))
idx = pd.MultiIndex.from_tuples(pairs)

In [17]:
# ===== Q4 =====
# TASK: Create the DataFrame `agents` from `commission`, with `idx` as the index and the columns `"week1"` and `"week2
# Build the DataFrame
# Give the two index levels names
agents =  pd.DataFrame(commission, index=idx, columns=['week1', 'week2'])
agents.index.names = ['city', 'service']

agents

week1  week2
city    service                
Douala  MoMo          82     90
        Airtime       40     44
        Transfers     65     70
Yaounde MoMo          75     78
        Airtime       38     36
        Transfers     60     66

In [18]:
# (provided) a ready-made copy, used by the next questions so you never get stuck
agents_ok = pd.DataFrame(commission, index=pd.MultiIndex.from_tuples(list(zip(cities, services)), names=['city', 'service']), columns=['week1', 'week2'])
agents_ok

week1  week2
city    service                
Douala  MoMo          82     90
        Airtime       40     44
        Transfers     65     70
Yaounde MoMo          75     78
        Airtime       38     36
        Transfers     60     66

### Question 5  *(3 marks)*
**TASK:** Selecting with a MultiIndex on `agents_ok`: (a) all rows of **Douala** (`douala`); (b) only the **Airtime** row of Douala (`douala_air`) by chaining two `.loc`; (c) the single **week2** column (`wk2`).

> 💡 **HINT:** Slides 224–225: `df.loc['Douala']` gives the inner table; chain `.loc['Airtime']`. A column is selected with `df['week2']`.

In [19]:
# ===== Q5 =====
# TASK: Selecting with a MultiIndex on `agents_ok`: (a) all rows of **Douala** (`douala`); (b) only the **Airtime** ro
# douala     -> .loc on the first level
# douala_air -> chain a second .loc
# wk2        -> one column
douala = agents_ok.loc['Douala']
douala_air = agents_ok.loc['Douala'].loc['Airtime']
wk2 = agents_ok['week2']

# show your answers
print('douala'); display(douala)
print('douala_air'); display(douala_air)
print('wk2'); display(wk2)

douala


,week1,week2
service,,
MoMo,82,90
Airtime,40,44
Transfers,65,70


douala_air


week1    40
week2    44
Name: Airtime, dtype: int64

wk2


city     service  
Douala   MoMo         90
         Airtime      44
         Transfers    70
Yaounde  MoMo         78
         Airtime      36
         Transfers    66
Name: week2, dtype: int64

### Question 6  *(3 marks)*
**TASK:** Cross-sections with `.xs()` on `agents_ok`: (a) everything for **Yaounde** (`yde`); (b) the row (**Yaounde**, **MoMo**) (`yde_momo`); (c) the **Airtime** rows of *both* cities, selecting by the level name `"service"` (`airtime_all`).

> 💡 **HINT:** `df.xs(key)`, `df.xs((key1, key2))` and `df.xs(key, level='level_name')`. (`xs` = cross-section; see the pandas docs – it extends the MultiIndex slides 224–225.)

In [20]:
# ===== Q6 =====
# TASK: Cross-sections with `.xs()` on `agents_ok`: (a) everything for **Yaounde** (`yde`); (b) the row (**Yaounde**, 
# yde         -> xs with one key
# yde_momo    -> xs with a tuple of keys
# airtime_all -> xs with the level argument
yde = agents_ok.xs('Yaounde')
yde_momo = agents_ok.xs(('Yaounde', 'MoMo'))
airtime_all = agents_ok.xs('Airtime', level='service')

# show your answers
print('yde'); display(yde)
print('yde_momo'); display(yde_momo)
print('airtime_all'); display(airtime_all)

yde


,week1,week2
service,,
MoMo,75,78
Airtime,38,36
Transfers,60,66


yde_momo


week1    75
week2    78
Name: (Yaounde, MoMo), dtype: int64

airtime_all


,week1,week2
city,,
Douala,40,44
Yaounde,38,36


---
## Part C – Stacking branch sheets with `concat()`
Each month the shops send their sales sheet separately. You must stack January, February and March into one table. Run the cell to load them.

In [22]:
jan = pd.DataFrame({"shop": ["Akwa", "Bonamoussadi"], "product": ["SIM", "Phone"], "units": [120, 15]})
feb = pd.DataFrame({"shop": ["Akwa", "Bonamoussadi"], "product": ["SIM", "Phone"], "units": [135, 12]})
mar = pd.DataFrame({"shop": ["Akwa", "Bonamoussadi"], "product": ["SIM", "Phone"], "units": [150, 20]})
                    

### Question 7  *(2 marks)*
**TASK:** Stack the three sheets **one under another** (`quarter`). First keep the original row numbers (`quarter_raw`) and observe the repeated index; then fix it with a clean index (`quarter`).

> 💡 **HINT:** Slide 217: `pd.concat([df1, df2, df3])`. The argument `ignore_index=True` renumbers the rows 0..n-1.

In [23]:
# ===== Q7 =====
# TASK: Stack the three sheets **one under another** (`quarter`). First keep the original row numbers (`quarter_raw`) 
# quarter_raw -> concat of the three tables
# quarter     -> same but with ignore_index=True
quarter_raw =  pd.concat([jan, feb, mar])
quarter =  pd.concat([jan, feb, mar], ignore_index=True)

# show your answers
print('quarter_raw'); display(quarter_raw)
print('quarter'); display(quarter)

quarter_raw


,shop,product,units
0,Akwa,SIM,120
1,Bonamoussadi,Phone,15
0,Akwa,SIM,135
1,Bonamoussadi,Phone,12
0,Akwa,SIM,150
1,Bonamoussadi,Phone,20


quarter


,shop,product,units
0,Akwa,SIM,120
1,Bonamoussadi,Phone,15
2,Akwa,SIM,135
3,Bonamoussadi,Phone,12
4,Akwa,SIM,150
5,Bonamoussadi,Phone,20


### Question 8  *(3 marks)*
**TASK:** The April sheet came from a different shop manager who named a column `qty` instead of `units`. (a) Concatenate `jan` and `apr` and see the `NaN`s (`mixed`). (b) Rename the column in `apr` and concatenate again (`fixed`). (c) Concatenate `jan` and `apr` keeping only the columns they have **in common** (`common_only`) – use `join="inner"`.

> 💡 **HINT:** Slide 217: columns with different names create new columns filled with NaN. `rename(columns={...})` fixes the name; `join='inner'` keeps only shared columns.

In [24]:
apr = pd.DataFrame({"shop": ["Akwa", "Bonamoussadi"], "product": ["SIM", "Phone"], "qty": [160, 18]})

In [25]:
# ===== Q8 =====
# TASK: The April sheet came from a different shop manager who named a column `qty` instead of `units`. (a) Concatenat
# mixed       -> plain concat (ignore_index=True)
# fixed       -> rename 'qty' to 'units' first, then concat
# common_only -> concat with join='inner'
mixed = pd.concat([jan, apr], ignore_index=True)
apr_fixed = apr.rename(columns={'qty': 'units'})
fixed = pd.concat([jan, apr_fixed], ignore_index=True)
common_only = pd.concat([jan, apr], join='inner', ignore_index=True)


# show your answers
print('mixed'); display(mixed)
print('fixed'); display(fixed)
print('common_only'); display(common_only)

mixed


,shop,product,units,qty
0,Akwa,SIM,120.0,NaN
1,Bonamoussadi,Phone,15.0,NaN
2,Akwa,SIM,NaN,160.0
3,Bonamoussadi,Phone,NaN,18.0


fixed


,shop,product,units
0,Akwa,SIM,120
1,Bonamoussadi,Phone,15
2,Akwa,SIM,160
3,Bonamoussadi,Phone,18


common_only


,shop,product
0,Akwa,SIM
1,Bonamoussadi,Phone
2,Akwa,SIM
3,Bonamoussadi,Phone


### Question 9  *(2 marks)*
**TASK:** Place `jan` and `feb` **side by side** (columns next to each other) in `side`. What goes wrong with the column names?

> 💡 **HINT:** Slide 217: `pd.concat([...], axis=1)` joins horizontally, aligning on the index.

In [26]:
# ===== Q9 =====
# TASK: Place `jan` and `feb` **side by side** (columns next to each other) in `side`. What goes wrong with the column
# Concatenate along the columns axis
side =  pd.concat([jan, feb], axis=1)


side

,shop,product,units,shop,product,units
0,Akwa,SIM,120,Akwa,SIM,135
1,Bonamoussadi,Phone,15,Bonamoussadi,Phone,12


---
## Part D – Combining staff tables with `merge()`
Data from three departments of the start-up lives in different tables. Run the cell, then combine them *by matching a key column* like a SQL join.

In [27]:
staff = pd.DataFrame({
    "employee":   ["Julia", "Marie", "Adam", "Nicole"],
    "department": ["Data Science", "Web Development", "Data Science", "Cyber Security"],
    "year_joined": [2019, 2021, 2022, 2018]})
pay = pd.DataFrame({
    "employee": ["Nicole", "Adam", "Julia", "Kevin"],
    "salary":   [22000, 16000, 20000, 15000]})
managers = pd.DataFrame({
    "manager":    ["Abraham", "Joseph", "Kayra"],
    "department": ["Web Development", "Cyber Security", "Data Science"]})

### Question 10  *(2 marks)*
**TASK:** Join `staff` and `pay` on the common column `employee` using the default (inner) merge (`staff_pay`). Which employee disappears, and why?

> 💡 **HINT:** Slide 218: `pd.merge(left, right, on='key')`. By default only keys present in **both** tables are kept.

In [28]:
# ===== Q10 =====
# TASK: Join `staff` and `pay` on the common column `employee` using the default (inner) merge (`staff_pay`). Which em
# staff_pay -> merge staff and pay on the employee column
staff_pay = pd.merge(staff, pay, on='employee')

staff_pay

,employee,department,year_joined,salary
0,Julia,Data Science,2019,20000
1,Adam,Data Science,2022,16000
2,Nicole,Cyber Security,2018,22000


### Question 11  *(2 marks)*
**TASK:** HR wants to keep **everybody from `staff`** even if there is no salary yet (`keep_staff`), and, separately, **everybody in `pay`** even if they are not in `staff` (`keep_pay`).

> 💡 **HINT:** Slide 218: `how='left'` keeps all rows of the left table; `how='right'` keeps all rows of the right table. Missing matches become NaN.

In [29]:
# ===== Q11 =====
# TASK: HR wants to keep **everybody from `staff`** even if there is no salary yet (`keep_staff`), and, separately, **
# keep_staff -> how='left'
# keep_pay   -> how='right'
keep_staff = pd.merge(staff, pay, on='employee', how='left')
keep_pay = pd.merge(staff, pay, on='employee', how='right')

# show your answers
print('keep_staff'); display(keep_staff)
print('keep_pay'); display(keep_pay)

keep_staff


,employee,department,year_joined,salary
0,Julia,Data Science,2019,20000.0
1,Marie,Web Development,2021,NaN
2,Adam,Data Science,2022,16000.0
3,Nicole,Cyber Security,2018,22000.0


keep_pay


,employee,department,year_joined,salary
0,Nicole,Cyber Security,2018.0,22000
1,Adam,Data Science,2022.0,16000
2,Julia,Data Science,2019.0,20000
3,Kevin,NaN,NaN,15000


### Question 12  *(2 marks)*
**TASK:** Show **all** employees from both tables, matched where possible (`everyone`), using an **outer** merge.

> 💡 **HINT:** `how='outer'` keeps the union of the keys.

In [30]:
# ===== Q12 =====
# TASK: Show **all** employees from both tables, matched where possible (`everyone`), using an **outer** merge.
# everyone -> outer merge on employee
everyone = pd.merge(staff, pay, on='employee', how='outer')

everyone

,employee,department,year_joined,salary
0,Adam,Data Science,2022.0,16000.0
1,Julia,Data Science,2019.0,20000.0
2,Kevin,NaN,NaN,15000.0
3,Marie,Web Development,2021.0,NaN
4,Nicole,Cyber Security,2018.0,22000.0


### Question 13  *(3 marks)*
**TASK:** Chain two merges: first `staff_pay` (inner merge of staff and pay), then add each department's **manager** by merging with `managers` on `department` (`full`).

> 💡 **HINT:** A merge returns a DataFrame, so you can merge it again. Key this time: `department`.

In [31]:
# ===== Q13 =====
# TASK: Chain two merges: first `staff_pay` (inner merge of staff and pay), then add each department's **manager** by 
# step 1: staff_pay_2 = merge staff & pay on employee
# step 2: full = merge staff_pay_2 with managers on department
staff_pay_2 = pd.merge(staff, pay, on='employee')
full = pd.merge(staff_pay_2, managers, on='department')

full

,employee,department,year_joined,salary,manager
0,Julia,Data Science,2019,20000,Kayra
1,Adam,Data Science,2022,16000,Kayra
2,Nicole,Cyber Security,2018,22000,Joseph


In [32]:
# (provided) a second table of employees from the Buea office – same column names, some overlap
buea = pd.DataFrame({
    "employee":   ["Julia", "Alex", "Adam", "Kayra"],
    "department": ["Data Science", "Web Development", "Data Science", "Cyber Security"],
    "university": ["Buea", "Douala", "Yaounde I", "Dschang"]})

### Question 14  *(3 marks)*
**TASK:** Merge `staff` and `buea` on **two keys at once** – `employee` AND `department` – with an inner merge (`both_keys`), then an outer merge (`both_outer`).

> 💡 **HINT:** Pass a list: `on=['employee', 'department']`. A row matches only if BOTH values match.

In [34]:
# ===== Q14 =====
# TASK: Merge `staff` and `buea` on **two keys at once** – `employee` AND `department` – with an inner merge (`both_ke
# both_keys  -> inner merge on the two columns
# both_outer -> outer merge on the two columns
both_keys = pd.merge(staff, buea, on=['employee', 'department'], how='inner')
#both_outer = pd.merge(staff, buea, on=['employee', 'department'], how='outer')

# show your answers
print('both_keys'); display(both_keys)
#print('both_outer'); display(both_outer)

both_keys


,employee,department,year_joined,university
0,Julia,Data Science,2019,Buea
1,Adam,Data Science,2022,Yaounde I


In [35]:
# ===== Q14 =====
# TASK: Merge `staff` and `buea` on **two keys at once** – `employee` AND `department` – with an inner merge (`both_ke
# both_keys  -> inner merge on the two columns
# both_outer -> outer merge on the two columns
#both_keys = pd.merge(staff, buea, on=['employee', 'department'], how='inner')
both_outer = pd.merge(staff, buea, on=['employee', 'department'], how='outer')

# show your answers
#print('both_keys'); display(both_keys)
print('both_outer'); display(both_outer)

both_outer


,employee,department,year_joined,university
0,Adam,Data Science,2022.0,Yaounde I
1,Alex,Web Development,NaN,Douala
2,Julia,Data Science,2019.0,Buea
3,Kayra,Cyber Security,NaN,Dschang
4,Marie,Web Development,2021.0,NaN
5,Nicole,Cyber Security,2018.0,NaN


---
## Part E – `join()`: combining on the index
`join()` is a shortcut for merging on the **index**. Two small tables of office equipment, labelled by room code, are given.

In [36]:
left  = pd.DataFrame({"X": ["X0", "X1", "X2"], "Y": ["Y0", "Y1", "Y2"]}, index=["A0", "A1", "A2"])
right = pd.DataFrame({"Z": ["Z0", "Z2", "Z3"], "T": ["T0", "T2", "T3"]}, index=["A0", "A2", "A3"])

### Question 15  *(3 marks)*
**TASK:** Join `right` to `left` with the default join (`j_default`), then an **inner** join (`j_inner`) and an **outer** join (`j_outer`).

> 💡 **HINT:** Slide 219: `left.join(right, how=...)`. The default is a *left* join.

In [42]:
# ===== Q15 =====
# TASK: Join `right` to `left` with the default join (`j_default`), then an **inner** join (`j_inner`) and an **outer*
# j_default -> left.join(right)
# j_inner   -> how='inner'
# j_outer   -> how='outer'
j_default = left.join(right)
j_inner = left.join(right, how='inner')
j_outer = left.join(right, how='outer')


# show your answers
print('j_default'); display(j_default)
print('j_inner'); display(j_inner)
print('j_outer'); display(j_outer)

j_default


,X,Y,Z,T
A0,X0,Y0,Z0,T0
A1,X1,Y1,NaN,NaN
A2,X2,Y2,Z2,T2


j_inner


,X,Y,Z,T
A0,X0,Y0,Z0,T0
A2,X2,Y2,Z2,T2


j_outer


,X,Y,Z,T
A0,X0,Y0,Z0,T0
A1,X1,Y1,NaN,NaN
A2,X2,Y2,Z2,T2
A3,NaN,NaN,Z3,T3


In [43]:
# (provided) two tables sharing the column name "Y" and a "key" column (not an index!)
t1 = pd.DataFrame({"key": ["X0", "X2", "X3", "X4"], "Y": ["Y0", "Y2", "Y3", "Y4"]})
t2 = pd.DataFrame({"key": ["X0", "X2", "X3"], "Z": ["Z0", "Z2", "Z3"]})

### Question 16  *(3 marks)*
**TASK:** (a) Make `key` the index of both tables, then join them (`keyed`). (b) Why can't you just run `t1.join(t2)`? Try it with suffixes `lsuffix='_left'`, `rsuffix='_right'` (`suffixed`).

> 💡 **HINT:** `join` works on the index, so call `set_index('key')` first (Part A!). If two tables share a column name, `join` needs `lsuffix=` / `rsuffix=`.

In [44]:
# ===== Q16 =====
# TASK: (a) Make `key` the index of both tables, then join them (`keyed`). (b) Why can't you just run `t1.join(t2)`? T
# keyed    -> set_index('key') on both, then join
# suffixed -> t1.join(t2, lsuffix=..., rsuffix=...)
keyed = t1.set_index('key').join(t2.set_index('key'))
suffixed = t1.join(t2, lsuffix='_left', rsuffix='_right')
# The two tables both contain 'key'; suffixes give these columns distinct names.
# show your answers
print('keyed'); display(keyed)
print('suffixed'); display(suffixed)

keyed


,Y,Z
key,,
X0,Y0,Z0
X2,Y2,Z2
X3,Y3,Z3
X4,Y4,NaN


suffixed


,key_left,Y,key_right,Z
0,X0,Y0,X0,Z0
1,X2,Y2,X2,Z2
2,X3,Y3,X3,Z3
3,X4,Y4,NaN,NaN


---
## Part F – Summarising cocoa deliveries (aggregation & grouping)
A cocoa union collected **60 deliveries** (2019-2024) from five cooperatives. The table `cocoa_df` has: `coop`, `grade` (A/B/C), `year`, `kg` delivered, `price_kg` (FCFA) and `transport_km` (distance to the warehouse – sometimes **missing**). Run the cell and use `cocoa_df`.

In [45]:
rng = np.random.default_rng(2026)
coops = ["Bokito", "Mbalmayo", "Ebolowa", "Kumba", "Mamfe"]
cocoa_df = pd.DataFrame({
    "coop":  [coops[i % 5] for i in range(60)],
    "grade": [["A", "B", "C"][(i // 5) % 3] for i in range(60)],
    "year":  [2019 + (i // 10) for i in range(60)],
    "kg":    rng.integers(200, 900, 60),
    "price_kg": rng.integers(900, 1600, 60),
    "transport_km": rng.integers(15, 160, 60).astype(float)})
cocoa_df.loc[[3, 11, 19, 27, 35, 44, 52], "transport_km"] = np.nan
cocoa_df.head()

,coop,grade,year,kg,price_kg,transport_km
0,Bokito,A,2019,796,1575,141.0
1,Mbalmayo,A,2019,325,1046,90.0
2,Ebolowa,A,2019,218,1554,124.0
3,Kumba,A,2019,647,1512,NaN
4,Mamfe,A,2019,455,1569,120.0


### Question 17  *(2 marks)*
**TASK:** Explore the table: store its `shape` (`c_shape`), the first 5 rows (`c_head`) and the last 5 rows (`c_tail`). Then run `cocoa_df.info()` and notice which column has fewer than 60 non-null values.

> 💡 **HINT:** Slides 137–141: `.shape`, `.head()`, `.tail()` and `.info()` for data types and non-null counts.

In [46]:
# ===== Q17 =====
# TASK: Explore the table: store its `shape` (`c_shape`), the first 5 rows (`c_head`) and the last 5 rows (`c_tail`). 
# c_shape, c_head, c_tail
# then call .info() on its own line
c_shape = cocoa_df.shape
c_head = cocoa_df.head()
c_tail = cocoa_df.tail()
# show your answers
print('c_shape'); display(c_shape)
print('c_head'); display(c_head)
print('c_tail'); display(c_tail)
cocoa_df.info()

c_shape


(60, 6)

c_head


,coop,grade,year,kg,price_kg,transport_km
0,Bokito,A,2019,796,1575,141.0
1,Mbalmayo,A,2019,325,1046,90.0
2,Ebolowa,A,2019,218,1554,124.0
3,Kumba,A,2019,647,1512,NaN
4,Mamfe,A,2019,455,1569,120.0


c_tail


,coop,grade,year,kg,price_kg,transport_km
55,Bokito,C,2024,616,1191,102.0
56,Mbalmayo,C,2024,428,1457,82.0
57,Ebolowa,C,2024,504,901,96.0
58,Kumba,C,2024,664,927,98.0
59,Mamfe,C,2024,409,1455,144.0


<class 'pandas.DataFrame'>
RangeIndex: 60 entries, 0 to 59
Data columns (total 6 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   coop          60 non-null     str    
 1   grade         60 non-null     str    
 2   year          60 non-null     int64  
 3   kg            60 non-null     int64  
 4   price_kg      60 non-null     int64  
 5   transport_km  53 non-null     float64
dtypes: float64(1), int64(3), str(2)
memory usage: 3.4 KB


### Question 18  *(3 marks)*
**TASK:** Calculate on the column `kg`: the mean (`kg_mean`), median (`kg_median`), minimum (`kg_min`), maximum (`kg_max`), standard deviation (`kg_std`), variance (`kg_var`) and the grand total (`kg_sum`).

> 💡 **HINT:** Slides 145–148: `.mean()`, `.median()`, `.min()`, `.max()`, `.std()`, `.var()`, `.sum()` – all methods of a column.

In [47]:
# ===== Q18 =====
# TASK: Calculate on the column `kg`: the mean (`kg_mean`), median (`kg_median`), minimum (`kg_min`), maximum (`kg_max
# One statistic per variable, each from the column cocoa_df['kg']
kg_mean = cocoa_df['kg'].mean()
kg_median = cocoa_df['kg'].median()
kg_min = cocoa_df['kg'].min()
kg_max = cocoa_df['kg'].max()
kg_std = cocoa_df['kg'].std()
kg_var = cocoa_df['kg'].var()
kg_sum = cocoa_df['kg'].sum()

# show your answers
print('kg_mean'); display(kg_mean)
print('kg_median'); display(kg_median)
print('kg_min'); display(kg_min)
print('kg_max'); display(kg_max)
print('kg_std'); display(kg_std)
print('kg_var'); display(kg_var)
print('kg_sum'); display(kg_sum)

kg_mean


np.float64(537.7333333333333)

kg_median


np.float64(524.0)

kg_min


np.int64(208)

... (4 outputs left out)

kg_var


np.float64(35791.11412429379)

kg_sum


np.int64(32264)

### Question 19  *(2 marks)*
**TASK:** Get all the numeric summary statistics at once for the **columns `kg`, `price_kg` and `transport_km`** (`summary`), and then flip the table so each variable is a **row** (`summary_t`).

> 💡 **HINT:** Slides 142–145: `describe()`. `.T` (transpose) swaps rows and columns.

In [48]:
# ===== Q19 =====
# TASK: Get all the numeric summary statistics at once for the **columns `kg`, `price_kg` and `transport_km`** (`summa
# summary   -> describe() on the three columns
# summary_t -> transpose of summary
summary = cocoa_df[['kg', 'price_kg', 'transport_km']].describe()
summary_t = summary.T

# show your answers
print('summary'); display(summary)
print('summary_t'); display(summary_t)

summary


,kg,price_kg,transport_km
count,60.000000,60.000000,53.000000
mean,537.733333,1257.666667,83.584906
std,189.185396,194.398391,40.983315
min,208.000000,901.000000,16.000000
25%,392.750000,1136.500000,54.000000
50%,524.000000,1245.500000,86.000000
75%,671.250000,1413.250000,116.000000
max,893.000000,1580.000000,157.000000


summary_t


,count,mean,std,min,25%,50%,75%,max
kg,60.0,537.733333,189.185396,208.0,392.75,524.0,671.25,893.0
price_kg,60.0,1257.666667,194.398391,901.0,1136.50,1245.5,1413.25,1580.0
transport_km,53.0,83.584906,40.983315,16.0,54.00,86.0,116.00,157.0


### Question 20  *(3 marks)*
**TASK:** Categorical overview: (a) how many deliveries per cooperative (`per_coop`); (b) the **unique** grades (`grades`); (c) the **number of unique** years (`n_years`); (d) how many `transport_km` values are missing (`n_missing`).

> 💡 **HINT:** Slides 149–154: `value_counts()`, `unique()`, `nunique()`; slide 101: `isnull().sum()`.

In [49]:
# ===== Q20 =====
# TASK: Categorical overview: (a) how many deliveries per cooperative (`per_coop`); (b) the **unique** grades (`grades
# per_coop -> value_counts of the coop column
# grades   -> unique values of grade
# n_years  -> nunique of year
# n_missing-> isnull().sum() on transport_km
per_coop = cocoa_df['coop'].value_counts()
grades = cocoa_df['grade'].unique()
n_years = cocoa_df['year'].nunique()
n_missing = cocoa_df['transport_km'].isnull().sum()

# show your answers
print('per_coop'); display(per_coop)
print('grades'); display(grades)
print('n_years'); display(n_years)
print('n_missing'); display(n_missing)

per_coop


coop
Bokito      12
Mbalmayo    12
Ebolowa     12
Kumba       12
Mamfe       12
Name: count, dtype: int64

grades


<ArrowStringArray>
['A', 'B', 'C']
Length: 3, dtype: str

n_years


6

n_missing


np.int64(7)

### Question 21  *(2 marks)*
**TASK:** Which cooperative delivers the most? Compute the **average `kg` per cooperative** two ways: as a Series (`avg_series`, using a single bracket) and as a one-column DataFrame (`avg_frame`, using double brackets).

> 💡 **HINT:** Slides 129–130: `df.groupby('col')['target'].mean()` returns a Series; `df.groupby('col')[['target']].mean()` returns a DataFrame.

In [50]:
# ===== Q21 =====
# TASK: Which cooperative delivers the most? Compute the **average `kg` per cooperative** two ways: as a Series (`avg_
# avg_series -> groupby coop, select 'kg', mean
# avg_frame  -> same but with [['kg']]
avg_series = cocoa_df.groupby('coop')['kg'].mean()
avg_frame = cocoa_df.groupby('coop')[['kg']].mean()

# show your answers
print('avg_series'); display(avg_series)
print('avg_frame'); display(avg_frame)

avg_series


coop
Bokito      620.500000
Ebolowa     483.750000
Kumba       530.416667
Mamfe       474.750000
Mbalmayo    579.250000
Name: kg, dtype: float64

avg_frame


,kg
coop,
Bokito,620.500000
Ebolowa,483.750000
Kumba,530.416667
Mamfe,474.750000
Mbalmayo,579.250000


### Question 22  *(2 marks)*
**TASK:** How did the **total kg delivered** evolve per `year` (`kg_per_year`)? Then isolate only the deliveries of the year **2023** (`y2023`).

> 💡 **HINT:** `groupby('year')[['kg']].sum()` for the first; slide 87 filtering for the second.

In [51]:
# ===== Q22 =====
# TASK: How did the **total kg delivered** evolve per `year` (`kg_per_year`)? Then isolate only the deliveries of the 
# kg_per_year -> groupby year, sum of kg
# y2023       -> filter rows where year equals 2023
kg_per_year =  cocoa_df.groupby('year')[['kg']].sum()
y2023 = cocoa_df[cocoa_df['year'] == 2023]

# show your answers
print('kg_per_year'); display(kg_per_year)
print('y2023'); display(y2023)

kg_per_year


,kg
year,
2019,4780
2020,6220
2021,5975
2022,4924
2023,5211
2024,5154


y2023


,coop,grade,year,kg,price_kg,transport_km
40,Bokito,C,2023,797,1215,157.0
41,Mbalmayo,C,2023,568,1335,55.0
42,Ebolowa,C,2023,264,924,29.0
43,Kumba,C,2023,501,1160,112.0
44,Mamfe,C,2023,389,1151,NaN
45,Bokito,A,2023,664,1194,85.0
46,Mbalmayo,A,2023,781,1420,27.0
47,Ebolowa,A,2023,208,1246,28.0
48,Kumba,A,2023,526,1499,67.0
49,Mamfe,A,2023,513,1228,16.0


### Question 23  *(5 marks)*
**TASK:** 🏆 **Challenge.** For each **cooperative**, compute in ONE command the mean and maximum of `price_kg` and the sum of `kg` (`coop_stats`). Hint: `agg` accepts a dictionary of column -> list of functions. Then show the full descriptive statistics of `transport_km` per cooperative (`km_desc`).

> 💡 **HINT:** Slide 130–131: `.agg({'price_kg': ['mean', 'max'], 'kg': 'sum'})`; slide 144 for `describe()` combined with `groupby`.

In [52]:
# ===== Q23 =====
# TASK: 🏆 **Challenge.** For each **cooperative**, compute in ONE command the mean and maximum of `price_kg` and the s
# coop_stats -> groupby coop then agg with a dictionary
# km_desc    -> groupby coop, select transport_km, then describe()
coop_stats =  cocoa_df.groupby('coop').agg({'price_kg': ['mean', 'max'], 'kg': 'sum'})
km_desc = cocoa_df.groupby('coop')['transport_km'].describe()

# show your answers
print('coop_stats'); display(coop_stats)
print('km_desc'); display(km_desc)

coop_stats


price_kg          kg
                 mean   max   sum
coop                             
Bokito    1213.333333  1575  7446
Ebolowa   1301.000000  1554  5805
Kumba     1237.166667  1580  6365
Mamfe     1264.416667  1569  5697
Mbalmayo  1272.416667  1562  6951

km_desc


,count,mean,std,min,25%,50%,75%,max
coop,,,,,,,,
Bokito,11.0,109.636364,34.485570,66.0,82.00,102.0,139.50,157.0
Ebolowa,10.0,66.900000,40.951055,18.0,28.25,62.0,100.50,124.0
Kumba,11.0,75.636364,42.949442,20.0,39.00,75.0,105.00,152.0
Mamfe,10.0,84.600000,50.927399,16.0,44.50,76.5,137.25,146.0
Mbalmayo,11.0,79.727273,27.291357,27.0,57.00,90.0,99.50,116.0


---
### ✅ End of Part 2
Total marks available: **58**. Restart the kernel and run **all** cells top-to-bottom (`Kernel ▸ Restart & Run All`) before submitting – your notebook must run without errors.